In [1]:
from huggingface_hub import login
login()

In [ ]:
import modeling_selfcheck
import modeling_selfcheck_apiprompt
import torch
import spacy
import warnings
from openai import OpenAI
import os
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
selfcheck_unigram = modeling_selfcheck.SelfCheckNgram(n=1)
selfcheck_bertscore = modeling_selfcheck.SelfCheckBERTScore(rescale_with_baseline=True)
selfcheck_nli = modeling_selfcheck.SelfCheckNLI(device=device)
selfcheck_mqag = modeling_selfcheck.SelfCheckMQAG(device=device)
llm_model = "mistralai/Mistral-7B-Instruct-v0.2"
selfcheck_prompt = modeling_selfcheck.SelfCheckLLMPrompt(llm_model, device)
selfcheck_apiprompt = modeling_selfcheck_apiprompt.SelfCheckAPIPrompt(client_type="openai", 
                                                                      client=OpenAI(api_key=os.environ["UPSTAGE_API_KEY"], 
                                                                                    base_url="https://api.upstage.ai/v1/solar"), 
                                                                        model="solar-pro")

In [ ]:
import json
with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))

In [ ]:
test_data = dataset[0]
passage = test_data["gpt3_text"]
nlp = spacy.load("en_core_web_sm")
sentences = [sent.text.strip() for sent in nlp(passage).sents]
print("Sentences: ", sentences)

samples = test_data["gpt3_text_samples"]
print("Length of samples: ", len(samples))

In [ ]:
sent_scores_unigram = selfcheck_unigram.predict(
    sentences=sentences,
    passage=passage,
    sampled_passages=samples
)
print(sent_scores_unigram)

In [ ]:
sent_scores_bertscore = selfcheck_bertscore.predict(
    sentences=sentences,
    sampled_passages=samples,
)
print(sent_scores_bertscore)

In [ ]:
sent_scores_nli = selfcheck_nli.predict(
    sentences = sentences,
    sampled_passages=samples
)
print(sent_scores_nli)

In [9]:
sent_scores_mqag = selfcheck_mqag.predict(
    sentences=sentences,
    passage=passage,
    sampled_passages=samples,
    num_questions_per_sent=5,
    scoring_method='bayes_with_alpha',
    beta1=0.8, beta2=0.8
)
print(sent_scores_mqag)

In [10]:
sent_scores_prompt = selfcheck_prompt.predict(
    sentences=sentences,
    sampled_passages=samples,
    verbose=True
)
print(sent_scores_prompt)

In [ ]:
sent_scores_prompt_api = selfcheck_apiprompt.predict(
    sentences=sentences,
    sampled_passages=samples,
    verbose=True
)
print(sent_scores_prompt_api)